# AskSight — Run B + LoRA + stronger vision encoder (Colab / Kaggle GPU)

`photo → CLIP ViT-L/14 (frozen) → trained connector → Qwen2.5 + LoRA → answer`

This experiment keeps Run B's sentence targets, VQA + COCO description data, split, and evaluation. It enables LoRA on Qwen and swaps CLIP ViT-B/32 for the larger CLIP ViT-L/14 vision encoder. This tests both changes together; if the result improves, separate follow-up runs are needed to tell which change helped most.

## Quick start

1. Upload this notebook to Colab or Kaggle. Enable Internet and a GPU; an L4 or better is recommended. ViT-L/14 uses more memory and produces 257 image tokens, versus 50 with ViT-B/32.
2. Ensure the `dev` branch contains the AskSight `model/` package, then run cells in order.
3. Use a fresh runtime and keep the same VQA counts, seed, answer style, and evaluation set as Run B.
4. The trainer saves the checkpoint at the end. Download the zip before the runtime closes.

This is a controlled candidate, not a guarantee of higher accuracy. Review the evaluated answers and category-level results before choosing it.


## 0. Configuration

This combines two changes on top of Run B: LoRA rank 16 on Qwen and CLIP ViT-L/14 instead of ViT-B/32. Everything else stays aligned with Run B.

- Same 20,000/1,000/1,000 VQA split, sentence targets, 2,000 caption-training images, seed, and evaluation files.
- The larger encoder creates 257 visual tokens, so batch size is set to 2 to reduce memory use. If you get CUDA out-of-memory errors, use batch size 1 or a larger GPU.
- Use the same test set and inspect answer correctness, not just sentence style or loss.


In [ ]:
VERSION = "v0.5-b-lora16-clip-vit-large-patch14"
TRAIN, VAL, TEST = 20_000, 1_000, 1_000
CAPTION_IMAGES, CAPTION_VAL_IMAGES = 2000, 200
CAPTION_SENTENCES = 0
ANSWER_STYLE = "sentence"
LORA_RANK = 16
DTYPE = "float16"
SEED = 0
BATCH_SIZE = 2  # lower to 1 if the GPU runs out of memory
VISION_MODEL = "openai/clip-vit-large-patch14"
VISION_REVISION = "e073d58e5fea2f33477c3561de63fc5da34697ef"


## 1. Install

Clones the AskSight `dev` branch and installs the model package, including LoRA support. The pinned large vision model downloads when the model is first loaded.


In [ ]:
import os
import subprocess
import sys
import importlib.metadata
from packaging.version import Version
REPO = "https://github.com/Pranjal6955/AskSight.git"
BRANCH = "dev"
WORKSPACE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(WORKSPACE, "AskSight-training-dev")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
%cd {REPO_DIR}/model
for required in (
    "asksight_model/robustness.py",
    "asksight_model/coco_captions.py",
    "asksight_model/evaluate_vlm.py",
    "evaluations/asksight_v1.jsonl",
    "asksight_model/targets.py",
):
    assert os.path.isfile(required), (
        f"Missing {required} on {BRANCH}. Push the latest AskSight changes to GitHub, "
        "restart this runtime, and rerun the notebook."
    )
!pip install -q -e ".[dev,vlm_eval,lora]"

# Kaggle images can ship torchao 0.10, which recent PEFT detects but cannot use.
# AskSight LoRA does not require torchao, so remove only incompatible versions.
try:
    torchao_version = Version(importlib.metadata.version("torchao"))
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and torchao_version < Version("0.16.0"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    print(f"Removed incompatible optional torchao {torchao_version}; standard PEFT LoRA will be used.")


## 2. Verify GPU

Asserts CUDA is available; prints the assigned GPU (expect `Tesla T4`). If it fails: **Runtime → Change runtime type → T4 GPU**, then re-run.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook Settings first"
import asksight_model
print("asksight_model", asksight_model.__name__, "ok")


## 3. Download VQA data

Downloads official VQA v2 questions/annotations plus the COCO images needed for this split. The split is deterministic and image-disjoint: images in train are kept separate from validation and test. The fingerprints are saved in `data/vqa/meta.json`.

For 50k, change the counts in cell 2 before running this cell. Actual counts can be slightly higher because the splitter keeps all questions for selected images together.


In [ ]:
%%time
!python -m asksight_model.data --out data/vqa --train {TRAIN} --val {VAL} --test {TEST} --seed {SEED} --answer-style {ANSWER_STYLE}
!cat data/vqa/meta.json


## 3b. Add COCO descriptions to training

The next cell samples 2,000 COCO train2014 images for training and 200 different images for caption validation. It combines up to `CAPTION_SENTENCES` distinct captions per image into `Describe this photo.` targets and downloads only selected images. The training and validation image IDs are disjoint from each other and the VQA validation/test image IDs. This is a first mixed-data candidate, not evidence that descriptions are grounded on AskSight's real user photos.

Set `EXTRA_EXAMPLES_FILE` to an optional reviewed JSONL (OCR/signs, unanswerable photos, real AskSight questions) with `question_id`, `image_id`, `image`, `question`, and `answer`. Images must be uploaded under `data/vqa/`. Do not include validation/test images.


In [ ]:
# Mix VQA questions with COCO descriptions (and optional reviewed examples). Safe to rerun.
import hashlib
import json
import subprocess
from pathlib import Path

CUSTOM_EXAMPLES_FILE = "data/asksight_coco_captions.jsonl"
CAPTION_VAL_FILE = "data/asksight_coco_captions_val.jsonl"
EXTRA_EXAMPLES_FILE = ""  # optional reviewed OCR/abstain/real-photo JSONL
prep_result = subprocess.run([
    "python", "-m", "asksight_model.coco_captions",
    "--data-dir", "data/vqa",
    "--out", CUSTOM_EXAMPLES_FILE,
    "--num-images", str(CAPTION_IMAGES),
    "--val-images", str(CAPTION_VAL_IMAGES),
    "--val-out", CAPTION_VAL_FILE,
    "--sentences-per-image", str(CAPTION_SENTENCES),
    "--seed", str(SEED),
], text=True, capture_output=True)
if prep_result.stdout:
    print(prep_result.stdout)
if prep_result.returncode:
    if prep_result.stderr:
        print(prep_result.stderr)
    raise subprocess.CalledProcessError(
        prep_result.returncode, prep_result.args, output=prep_result.stdout, stderr=prep_result.stderr
    )

examples_path = Path(CUSTOM_EXAMPLES_FILE)
caption_val_path = Path(CAPTION_VAL_FILE)
if not examples_path.is_file() or not caption_val_path.is_file():
    raise FileNotFoundError("COCO caption preparation did not produce both JSONL files")

data_dir = Path("data/vqa")
train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"
records = [json.loads(line) for line in train_path.read_text().splitlines() if line.strip()]
records = [
    row for row in records
    if row.get("source") not in {"asksight_coco_caption_train", "asksight_custom_train"}
]
vqa_train_images = {str(record["image_id"]) for record in records}
val_records = [json.loads(line) for line in val_path.read_text().splitlines() if line.strip()]
val_records = [row for row in val_records if row.get("source") != "asksight_coco_caption_val"]
held_out_images = {str(row["image_id"]) for row in val_records}
test_records = [json.loads(line) for line in (data_dir / "test.jsonl").read_text().splitlines() if line.strip()]
held_out_images |= {str(row["image_id"]) for row in test_records}
train_caption_rows = [
    json.loads(line) for line in examples_path.read_text().splitlines() if line.strip()
]
val_caption_rows = [
    json.loads(line) for line in caption_val_path.read_text().splitlines() if line.strip()
]
extra_rows = []
if EXTRA_EXAMPLES_FILE:
    extra_path = Path(EXTRA_EXAMPLES_FILE)
    if not extra_path.is_file():
        raise FileNotFoundError(extra_path)
    extra_rows = [json.loads(line) for line in extra_path.read_text().splitlines() if line.strip()]
    for row in extra_rows:
        row["source"] = "asksight_custom_train"
train_caption_rows.extend(extra_rows)
train_caption_images = {str(row["image_id"]) for row in train_caption_rows}
val_caption_images = set()
existing_ids = {str(row["question_id"]) for row in records}
existing_ids |= {str(row["question_id"]) for row in val_records}
existing_ids |= {str(row["question_id"]) for row in test_records}

for row_number, row in enumerate(train_caption_rows, start=1):
    for field in ("question_id", "image_id", "image", "question", "answer"):
        if field not in row or not str(row[field]).strip():
            raise ValueError(f"train example {row_number} needs a non-empty {field!r}")
    row["question_id"] = str(row["question_id"])
    row["image_id"] = str(row["image_id"])
    if row["image_id"] in held_out_images or row["question_id"] in existing_ids:
        raise ValueError(f"train example {row_number} overlaps held-out data or question IDs")
    if not (data_dir / row["image"]).is_file():
        raise FileNotFoundError(f"train image not found: {data_dir / row['image']}")
    if row.get("source") != "asksight_custom_train":
        row["source"] = "asksight_coco_caption_train"
    existing_ids.add(row["question_id"])
    records.append(row)

for row_number, row in enumerate(val_caption_rows, start=1):
    row["question_id"] = str(row["question_id"])
    row["image_id"] = str(row["image_id"])
    if (
        row["image_id"] in held_out_images
        or row["image_id"] in vqa_train_images
        or row["image_id"] in train_caption_images
    ):
        raise ValueError(f"caption validation row {row_number} has image leakage")
    if row["question_id"] in existing_ids:
        raise ValueError(f"caption validation row {row_number} duplicates a question ID")
    if not (data_dir / row["image"]).is_file():
        raise FileNotFoundError(f"caption image not found: {data_dir / row['image']}")
    row["source"] = "asksight_coco_caption_val"
    existing_ids.add(row["question_id"])
    val_caption_images.add(row["image_id"])
    val_records.append(row)

train_path.write_text("".join(json.dumps(row) + "\n" for row in records))
val_path.write_text("".join(json.dumps(row) + "\n" for row in val_records))
meta_path = data_dir / "meta.json"
meta = json.loads(meta_path.read_text())
extra_path = Path(EXTRA_EXAMPLES_FILE) if EXTRA_EXAMPLES_FILE else None
meta["custom_examples"] = {
    "train_file": examples_path.name,
    "train_sha256": hashlib.sha256(examples_path.read_bytes()).hexdigest(),
    "added_to_train": len(train_caption_rows) - len(extra_rows),
    "extra_file": extra_path.name if extra_path else None,
    "extra_sha256": hashlib.sha256(extra_path.read_bytes()).hexdigest() if extra_path else None,
    "added_extra": len(extra_rows),
    "validation_file": caption_val_path.name,
    "validation_sha256": hashlib.sha256(caption_val_path.read_bytes()).hexdigest(),
    "added_to_val": len(val_caption_rows),
    "train_images": len(train_caption_images),
    "validation_images": len(val_caption_images),
}
meta["actual"]["train"] = len(records)
meta["actual"]["val"] = len(val_records)
for split_name, split_rows in (("train", records), ("val", val_records)):
    meta["split_fingerprints"][split_name] = hashlib.sha256(
        ",".join(sorted(str(row["question_id"]) for row in split_rows)).encode()
    ).hexdigest()
meta_path.write_text(json.dumps(meta, indent=2) + "\n")
print(
    f"Mixed data: {len(records)} train rows ({ANSWER_STYLE} targets, CAPTION_SENTENCES={CAPTION_SENTENCES}); "
    f"{len(val_records)} val rows"
)


## 4. Train the candidate

The generated config pins CLIP ViT-L/14 and sets batch size 2. The trainer adds rank-16 LoRA to Qwen and trains it with the connector. The vision encoder stays frozen, as in Run B. Training can take longer because each image contributes 257 visual tokens.


In [ ]:
%%time
import yaml
from pathlib import Path

with open("configs/default.yaml") as f:
    run_config = yaml.safe_load(f)
run_config["model"]["vision_model"] = VISION_MODEL
run_config["model"]["vision_revision"] = VISION_REVISION
run_config["model"]["lora_rank"] = LORA_RANK
run_config["train"]["batch_size"] = BATCH_SIZE
run_config["train"]["seed"] = SEED
run_config["train"]["version"] = VERSION
run_config["train"]["dtype"] = DTYPE
Path("configs/run_b_lora_strong_vision.yaml").write_text(yaml.safe_dump(run_config, sort_keys=False))

!rm -rf checkpoints/{VERSION}
!python -m asksight_model.train --config configs/run_b_lora_strong_vision.yaml --version {VERSION}


## 5. Evaluate the candidate

Use the same held-out VQA split and fixed AskSight questions as Run B. Compare answer correctness, sentence rate, and latency. The larger encoder may improve visual details but will likely be slower.


In [ ]:
%%time
from pathlib import Path

checkpoint_metrics = Path(f"checkpoints/{VERSION}/metrics.json")
if not checkpoint_metrics.is_file():
    raise FileNotFoundError(
        f"Training checkpoint is missing: {checkpoint_metrics}. "
        "Run cell 12 and wait for 'checkpoint saved to ...' before evaluating. "
        "If training failed, fix its earlier error first; evaluation cannot create metrics.json."
    )

!python -m asksight_model.evaluate --checkpoint checkpoints/{VERSION} --split test --data-dir data/vqa
import json
metrics = json.load(open(f"checkpoints/{VERSION}/metrics.json"))
if "eval_test" not in metrics:
    raise RuntimeError(f"training checkpoint {VERSION} is missing eval_test; see cell 14 output")
ev = metrics["eval_test"]
print(
    f"\nVQA exact {ev['vqa_accuracy']:.1%} | contains {ev['vqa_accuracy_contains']:.1%} | "
    f"sentence rate {ev['sentence_rate']:.1%} | latency median {ev['latency_median_s']}s, p95 {ev['latency_p95_s']}s"
)
!python -m asksight_model.evaluate --checkpoint checkpoints/{VERSION} --questions-file evaluations/asksight_v1.jsonl
metrics = json.load(open(f"checkpoints/{VERSION}/metrics.json"))
if "eval_asksight_asksight_v1" not in metrics:
    raise RuntimeError(f"checkpoint {VERSION} is missing fixed-set evaluation; rerun cell 14")
qs = metrics["eval_asksight_asksight_v1"]
print(f"AskSight: sentence rate {qs['sentence_rate']:.1%}, mean words {qs['mean_words']}")
if "fact_recall" in qs:
    print(f"description fact recall {qs['fact_recall']:.1%} (heuristic; review predictions)")


## 5a. Review 50 answers

Prints 25 random test answers plus all 26 fixed AskSight cases. For each answer, note: complete sentence, correct, invented details, and the error type (color, text, count, description).


In [ ]:
# Print 25 random test answers plus the 26 fixed AskSight cases.
import json
import random
from pathlib import Path

predictions_path = Path(f"checkpoints/{VERSION}/predictions_test.jsonl")
asksight_path = Path(f"checkpoints/{VERSION}/predictions_asksight_asksight_v1.jsonl")
missing = [str(path) for path in (predictions_path, asksight_path) if not path.is_file()]
if missing:
    raise FileNotFoundError(
        f"Evaluation predictions are missing: {', '.join(missing)}. "
        "Run cell 12 successfully, then cell 14 to generate predictions before this review cell."
    )

test = [json.loads(line) for line in predictions_path.open()]
fixed = [
    json.loads(line)
    for line in asksight_path.open()
]

random.seed(0)
for row in random.sample(test, 25) + fixed:
    print("Q:", row["question"])
    print("A:", row["prediction"])
    print("Ref:", row.get("short_answer") or row.get("references"))
    print("Facts:", row.get("expected_facts"))
    print("-" * 60)


## 5b. Compare with Run B

Run this after the original Run B checkpoint exists in this runtime. Compare both using the same data, split, and prompt. Review errors by question type (yes/no, count, object, text, description); this run changes LoRA and vision encoder together, so a separate follow-up experiment is needed to isolate their individual effects.


In [ ]:
# Compare Run B, Run C (LoRA only), and this combined candidate when checkpoints are available.
import json
from pathlib import Path

runs = {
    "Run B (sentence, connector only)": "v0.4-b-sentence",
    "Run C (sentence + LoRA 16, original vision)": "v0.4-c-lora16",
    "Run B + LoRA + ViT-L/14": VERSION,
}
print("v0.3 baseline: exact 49.97% | containment 51.05% | sentence rate 0%")
def _pct(value):
    return f"{value:.1%}" if isinstance(value, (int, float)) else "n/a"

prompts = set()
data_prints = set()
for label, version in runs.items():
    path = Path(f"checkpoints/{version}/metrics.json")
    if not path.is_file():
        print(f"\n{label}: missing {path}; run/download that checkpoint first")
        continue
    metrics = json.loads(path.read_text())
    test = metrics.get("eval_test")
    fixed = metrics.get("eval_asksight_asksight_v1")
    if test is None:
        print(f"\n{label} ({version}): missing eval_test; rerun cell 14")
        continue
    prompts.add(test.get("system_prompt_sha256"))
    mc = metrics.get("model_config", {})
    tc = metrics.get("train_config", {})
    data_meta = metrics.get("data_meta", {})
    fingerprints = data_meta.get("split_fingerprints", {})
    data_prints.add(fingerprints.get("test"))
    print(f"\n{label} ({version})")
    print(f"  vision: {mc.get('vision_model')} | lora_rank: {mc.get('lora_rank')} | batch: {tc.get('batch_size')} | dtype: {tc.get('dtype')} | device: {test.get('device')}")
    print(f"  test n: {test.get('n')} | train rows: {data_meta.get('actual', {}).get('train')} | test fingerprint: {str(fingerprints.get('test'))[:12]}")
    print(f"  sentence rate: {_pct(test.get('sentence_rate'))}")
    print(f"  exact VQA: {_pct(test.get('vqa_accuracy'))}")
    print(f"  containment: {_pct(test.get('vqa_accuracy_contains'))}")
    print(f"  mean words: {test.get('mean_words', 'n/a')}")
    print(f"  latency median/p95: {test.get('latency_median_s')}s / {test.get('latency_p95_s')}s")
    if fixed is None:
        print("  fixed: missing; rerun the fixed-set evaluation")
    else:
        print(f"  fixed sentence rate: {_pct(fixed.get('sentence_rate'))}")
        print(f"  fixed direct containment: {_pct(fixed.get('direct_vqa_accuracy_contains'))}")
if len(prompts) > 1:
    print("\nWARNING: the runs used different system prompts; comparison is not clean.")
if len(data_prints) > 1:
    print("\nWARNING: the runs used different VQA data splits; comparison is not clean.")
print("\nReview the same sampled answers and count hallucinations before selecting a model.")


## 5c. Pretrained vision-language baseline

Evaluate Qwen3-VL-2B-Instruct on the same fixed 26 cases. It is an evaluation-only baseline, not a drop-in checkpoint for the current backend. This directly checks whether a model pretrained to connect images and language improves the app tasks. It downloads several GB of model files.


In [ ]:
%%time
!python -m asksight_model.evaluate_vlm --questions-file evaluations/asksight_v1.jsonl --output evaluations/runs/{VERSION}-qwen3-vl-2b-asksight-v1.jsonl


## 5d. Robustness (~3 min)

Re-runs the same held-out split under one realistic capture fault per condition — `blur`, `dark`, `obstructed` (a finger over the lens), `lowres` — and reports the accuracy drop against `clean`. This is the measured basis for the "please verify this" guidance the app shows for poor photos. The result lands in `metrics.json` under `eval_robustness`.


In [ ]:
%%time
!python -m asksight_model.robustness --checkpoint checkpoints/{VERSION} --split test --data-dir data/vqa


## 6. Download the checkpoint

Packages `checkpoints/<VERSION>/`, including the connector and LoRA adapter. Colab storage is temporary, so download the archive before ending the session.


In [ ]:
!rm -f {VERSION}-experiment.zip
!zip -qr {VERSION}-experiment.zip checkpoints/{VERSION}
import os
import shutil
zip_path = f"{VERSION}-experiment.zip"
if os.path.isdir("/kaggle/working"):  # Kaggle may also expose Colab compatibility variables
    output_path = f"/kaggle/working/{VERSION}-experiment.zip"
    if os.path.abspath(zip_path) != os.path.abspath(output_path):
        shutil.copy2(zip_path, output_path)
    zip_path = output_path
    size = os.path.getsize(zip_path) / 1e6
    print(f"ready for Kaggle's Output panel: {os.path.abspath(zip_path)} ({size:.1f} MB)")
elif os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import files
    files.download(zip_path)
else:
    size = os.path.getsize(zip_path) / 1e6
    print(f"ready to download: {os.path.abspath(zip_path)} ({size:.1f} MB)")


## 7. Serve the checkpoint (laptop / server)

```bash
cd ~/AskSight-checkpoints && unzip ~/Downloads/{VERSION}-experiment.zip
cd backend && source venv/bin/activate
pip install -e "../model[lora]"   # include LoRA when the checkpoint uses it
export ASKSIGHT_MODEL_CHECKPOINT=$HOME/AskSight-checkpoints/checkpoints/{VERSION}
uvicorn app.main:app --reload   # /ask now returns real answers
```

For a larger experiment, set `TRAIN = 50000` and use a fresh `VERSION`. Compare on the same held-out data. Do not judge quality from answer length alone.
